# Handling Mixed-Type Variables

Some columns aren't cleanly numeric *or* categorical — they mix both in the same string (e.g. a ticket number that's sometimes just digits, sometimes a prefix plus digits). This notebook splits three such columns (`number`, `Cabin`, `Ticket`) into a clean numeric part and a clean categorical part each, so both halves can actually be used by a model.

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('titanic.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'titanic.csv'

In [ ]:
df.head()

## 2. The `number` Column — Mixed Numeric/Text Values

`.unique()` reveals the problem directly: some rows hold a plain number (family members traveling together), others hold a text code entirely — one column doing two jobs.

In [ ]:
df['number'].unique()

A quick visual of how common each value is:

In [ ]:
fig = df['number'].value_counts().plot.bar()
fig.set_title('Passengers travelling with')

### Splitting Into a Numeric Part and a Categorical Part

`pd.to_numeric(..., errors='coerce')` converts what it can into numbers, and turns anything it can't parse into `NaN` — which conveniently marks exactly the rows that were actually text codes.

In [ ]:
# extract numerical part
df['number_numerical'] = pd.to_numeric(df["number"],errors='coerce',downcast='integer')

`np.where(...)` then does the reverse for the categorical half: keep the original text **only** where the numeric conversion failed (i.e. where `number_numerical` is `NaN`), otherwise mark it as `NaN` too — the two new columns are complementary, together covering every row exactly once.

In [ ]:
# extract categorical part
df['number_categorical'] = np.where(df['number_numerical'].isnull(),df['number'],np.nan)

df.head()

### Checking the Split Worked

Both new columns should have missing values roughly summing to the total row count, confirming every original row landed in exactly one of the two new columns.

In [ ]:
print(df[['number_numerical', 'number_categorical']].isnull().sum())
print('Total rows:', len(df))


## 3. The `Cabin` Column — Letter Prefix + Number

Cabin codes like `'C123'` combine a **deck letter** (`C`) with a **cabin number** (`123`) — each likely carries different information (deck location vs. specific room), so it's worth splitting them out separately rather than treating the whole string as one category.

In [ ]:
df['Cabin'].unique()

## 4. The `Ticket` Column — Similarly Mixed

Some tickets are plain numbers; others have a prefix (e.g. `'PC 17599'`) — same mixed-type problem as `number`, just formatted differently.

In [ ]:
df['Ticket'].unique()

### Extracting Cabin's Numeric and Categorical Parts

`.str.extract(r'(\d+)')` pulls out the first run of digits found anywhere in the string (the cabin number); `.str[0]` grabs just the very first character (the deck letter).

In [ ]:
df['cabin_num'] = df['Cabin'].str.extract('(\d+)') # captures numerical part
df['cabin_cat'] = df['Cabin'].str[0] # captures the first letter

df.head()

Visualizing how common each deck letter is:

In [ ]:
df['cabin_cat'].value_counts().plot(kind='bar')

### Extracting Ticket's Numeric and Categorical Parts

A bit more involved than `Cabin`, since the prefix and number are **space-separated words** rather than glued together:
- `.split()[-1]` takes the **last** space-separated token (the number, which always comes last) and converts it with `pd.to_numeric(errors='coerce')`, same trick as `number` above.
- `.split()[0]` takes the **first** token as the candidate prefix — but for tickets that are *just* a number with no prefix, that "first token" is actually the number itself, so `np.where(..., str.isdigit(), NaN, ...)` cleans that up: if the extracted "prefix" is actually all digits, it isn't a real category, so it's replaced with `NaN`.

In [ ]:
# extract the last bit of ticket as number
df['ticket_num'] = df['Ticket'].apply(lambda s: s.split()[-1])
df['ticket_num'] = pd.to_numeric(df['ticket_num'],
                                   errors='coerce',
                                   downcast='integer')

# extract the first part of ticket as category
df['ticket_cat'] = df['Ticket'].apply(lambda s: s.split()[0])
df['ticket_cat'] = np.where(df['ticket_cat'].str.isdigit(), np.nan,
                              df['ticket_cat'])

df.head(20)

In [ ]:
df['ticket_cat'].unique()

### Visualizing Ticket Categories

In [ ]:
df['ticket_cat'].value_counts().plot(kind='bar', figsize=(10, 4))
plt.title('Ticket Prefix Categories')
plt.show()


## Summary

| Task | Method |
|---|---|
| Split a numeric-or-text column | `pd.to_numeric(col, errors='coerce')` — text becomes `NaN`, marking exactly which rows need the categorical half |
| Keep the text half only where numeric parsing failed | `np.where(numeric_col.isnull(), col, np.nan)` |
| Pull digits out of a glued string (e.g. `'C123'`) | `col.str.extract(r'(\d+)')` |
| Pull the first character (e.g. deck letter) | `col.str[0]` |
| Split a space-separated string, take one token | `col.apply(lambda s: s.split()[-1])` |
| Filter out "fake" categories that are actually all-digit | `np.where(col.str.isdigit(), np.nan, col)` |

### Why bother?
Left as one mixed column, neither the numeric information (family size, cabin/ticket number) nor the categorical information (deck, ticket prefix) is directly usable by most models — splitting them out turns one unusable column into two genuinely informative features.